## FT-transformer

## Model

In [ ]:
# ============================================================
# BUILT-IN FT-Transformer IMPLEMENTATION
# ============================================================

class FeatureTokenizer(nn.Module):
    """Tokenizes numerical and categorical features into embeddings"""
    def __init__(self, n_num_features, cat_cardinalities, d_token, numerical_bias=True, categorical_bias=True):
        super().__init__()
        self.n_num_features = n_num_features
        self.n_cat_features = len(cat_cardinalities)

        # Numerical features tokenization
        if self.n_num_features > 0:
            self.num_weight = nn.Parameter(torch.randn(n_num_features, d_token))
            if numerical_bias:
                self.num_bias = nn.Parameter(torch.zeros(d_token))
            else:
                self.register_parameter('num_bias', None)
            nn.init.kaiming_uniform_(self.num_weight, a=math.sqrt(5))

        # Categorical features tokenization
        if self.n_cat_features > 0:
            self.cat_embeddings = nn.ModuleList([
                nn.Embedding(card, d_token) for card in cat_cardinalities
            ])
            if categorical_bias:
                self.cat_bias = nn.Parameter(torch.zeros(d_token))
            else:
                self.register_parameter('cat_bias', None)

        self.norm = nn.LayerNorm(d_token)

    def forward(self, x_num, x_cat):
        """Convert numerical and categorical inputs to token embeddings"""
        tokens = []

        # Process numerical features: x_num * weight + bias
        if x_num is not None and self.n_num_features > 0:
            num_tokens = x_num.unsqueeze(-1) * self.num_weight.unsqueeze(0)
            if self.num_bias is not None:
                num_tokens = num_tokens + self.num_bias
            tokens.append(num_tokens)

        # Process categorical features: embed each category independently
        if x_cat is not None and self.n_cat_features > 0:
            cat_tokens = torch.stack([emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)], dim=1)
            if self.cat_bias is not None:
                cat_tokens = cat_tokens + self.cat_bias
            tokens.append(cat_tokens)

        if not tokens:
            raise ValueError("No input features provided")

        # Concatenate all tokens and apply layer normalization
        x = torch.cat(tokens, dim=1)
        return self.norm(x)


class FTTransformer(nn.Module):
    """FT-Transformer: Feature Tokenizer + Transformer for tabular data"""

    def __init__(self, n_num_features, cat_cardinalities, d_token=48, num_layers=3,
                 num_heads=4, dim_feedforward=192, dropout=0.1,
                 numerical_bias=True, categorical_bias=True):
        super().__init__()
        self.d_token = d_token

        # Feature tokenizer
        self.tokenizer = FeatureTokenizer(
            n_num_features=n_num_features,
            cat_cardinalities=cat_cardinalities,
            d_token=d_token,
            numerical_bias=numerical_bias,
            categorical_bias=categorical_bias
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output projection head for embeddings
        self.output_projection = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token),
            nn.GELU(),
            nn.Linear(d_token, d_token)
        )

        # Classification head (binary classifier)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1)
        )

    def forward(self, x_num, x_cat):
        """Forward pass: returns logits and embeddings"""
        tokens = self.tokenizer(x_num, x_cat)
        tokens = self.transformer(tokens)
        pooled = tokens.mean(dim=1)
        embeddings = self.output_projection(pooled)
        logits = self.classifier(embeddings).squeeze(-1)
        return logits, embeddings

    def get_embeddings(self, x_num, x_cat):
        """Extract embeddings (without classification head)"""
        tokens = self.tokenizer(x_num, x_cat)
        tokens = self.transformer(tokens)
        pooled = tokens.mean(dim=1)
        return self.output_projection(pooled)

## TRAINING AND EVALUATIO

In [ ]:
# ============================================================
# TRAINING AND EVALUATION FUNCTIONS
# ============================================================

def train_epoch_ft(model, loader, optimizer, criterion, device):
    """Train model for one epoch"""
    model.train()
    total_loss = 0
    for x_num, x_cat, y in loader:
        # Move data to device
        if x_num is not None:
            x_num = x_num.to(device)
        if x_cat is not None:
            x_cat = x_cat.to(device)
        y = y.to(device)

        # Forward and backward passes
        optimizer.zero_grad()
        logits, _ = model(x_num, x_cat)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_model_ft(model, loader, device):
    """Evaluate model and return classification metrics"""
    model.eval()
    all_preds = []
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for x_num, x_cat, y in loader:
            if x_num is not None:
                x_num = x_num.to(device)
            if x_cat is not None:
                x_cat = x_cat.to(device)

            logits, _ = model(x_num, x_cat)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(y.numpy())

    # Compute metrics
    recall = recall_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds)
    auprc = average_precision_score(all_targets, all_probs)
    roc_auc = roc_auc_score(all_targets, all_probs)

    return recall, f1, auprc, roc_auc


# ============================================================
# DATALOADER CREATION
# ============================================================

def create_ft_loaders(finetune_data, finetune_target, test_data, test_target, num_cols, cat_cols, batch_size=256):
    """Create DataLoaders for FT-Transformer"""

    # Prepare numerical features
    if len(num_cols) > 0:
        ft_num = torch.tensor(finetune_data[num_cols].values, dtype=torch.float32)
        test_num = torch.tensor(test_data[num_cols].values, dtype=torch.float32)
    else:
        ft_num = torch.zeros(len(finetune_target), 1, dtype=torch.float32)
        test_num = torch.zeros(len(test_target), 1, dtype=torch.float32)

    # Prepare categorical features
    if len(cat_cols) > 0:
        ft_cat = torch.tensor(finetune_data[cat_cols].values, dtype=torch.long)
        test_cat = torch.tensor(test_data[cat_cols].values, dtype=torch.long)
    else:
        ft_cat = torch.zeros(len(finetune_target), 1, dtype=torch.long)
        test_cat = torch.zeros(len(test_target), 1, dtype=torch.long)

    # Prepare targets
    ft_target = torch.tensor(finetune_target.values, dtype=torch.float32)
    test_target_tensor = torch.tensor(test_target.values, dtype=torch.float32)

    # Create datasets and loaders
    ft_dataset = TensorDataset(ft_num, ft_cat, ft_target)
    test_dataset = TensorDataset(test_num, test_cat, test_target_tensor)

    ft_loader = DataLoader(ft_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return ft_loader, test_loader


# ============================================================
# CROSS-VALIDATION
# ============================================================

def cross_validate_ft(finetune_data, finetune_target, num_cols, cat_cols, cat_cardinalities,
                      device, d_token=48, num_layers=3, num_heads=4,
                      dim_feedforward=192, dropout=0.1, batch_size=256, epochs=100):
    """Perform 5-fold stratified cross-validation for FT-Transformer"""

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    X_cv = finetune_data.copy()
    y_cv = finetune_target.copy()

    cv_recalls, cv_f1s, cv_auprcs, cv_rocs = [], [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(tqdm(skf.split(X_cv, y_cv), total=5, desc='CV Folds')):
        # Split data
        X_train = X_cv.iloc[train_idx]
        y_train = y_cv.iloc[train_idx]
        X_val = X_cv.iloc[val_idx]
        y_val = y_cv.iloc[val_idx]

        # Create fold-specific loaders
        if len(num_cols) > 0:
            train_num = torch.tensor(X_train[num_cols].values, dtype=torch.float32)
            val_num = torch.tensor(X_val[num_cols].values, dtype=torch.float32)
        else:
            train_num = torch.zeros(len(y_train), 1, dtype=torch.float32)
            val_num = torch.zeros(len(y_val), 1, dtype=torch.float32)

        if len(cat_cols) > 0:
            train_cat = torch.tensor(X_train[cat_cols].values, dtype=torch.long)
            val_cat = torch.tensor(X_val[cat_cols].values, dtype=torch.long)
        else:
            train_cat = torch.zeros(len(y_train), 1, dtype=torch.long)
            val_cat = torch.zeros(len(y_val), 1, dtype=torch.long)

        train_dataset = TensorDataset(train_num, train_cat, torch.tensor(y_train.values, dtype=torch.float32))
        val_dataset = TensorDataset(val_num, val_cat, torch.tensor(y_val.values, dtype=torch.float32))

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Initialize model
        model = FTTransformer(
            n_num_features=len(num_cols),
            cat_cardinalities=cat_cardinalities,
            d_token=d_token,
            num_layers=num_layers,
            num_heads=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        ).to(device)

        # Optimizer and loss
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
        pos_weight = (y_train == 0).sum() / (y_train == 1).sum() if (y_train == 1).sum() > 0 else 1.0
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))

        # Training with early stopping
        best_loss = float('inf')
        patience_counter = 0
        best_model_state = None

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for x_num, x_cat, y in train_loader:
                x_num = x_num.to(device) if len(num_cols) > 0 else None
                x_cat = x_cat.to(device) if len(cat_cols) > 0 else None
                y = y.to(device)

                optimizer.zero_grad()
                logits, _ = model(x_num, x_cat)
                loss = criterion(logits, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)

            # Early stopping logic
            if avg_loss < best_loss:
                best_loss = avg_loss
                best_model_state = model.state_dict().copy()
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= 20:
                break

        # Load best model and evaluate on validation set
        model.load_state_dict(best_model_state)

        # Оценка на валидации
        recall, f1, auprc, roc = evaluate_model_ft(model, val_loader, device)
        cv_recalls.append(recall)
        cv_f1s.append(f1)
        cv_auprcs.append(auprc)
        cv_rocs.append(roc)

    # Return mean and std of metrics across folds
    return {
        'recall': np.mean(cv_recalls), 'recall_std': np.std(cv_recalls),
        'f1': np.mean(cv_f1s), 'f1_std': np.std(cv_f1s),
        'auprc': np.mean(cv_auprcs), 'auprc_std': np.std(cv_auprcs),
        'roc': np.mean(cv_rocs), 'roc_std': np.std(cv_rocs)
    }


# ============================================================
# MAIN TRAINING
# ============================================================

def train_ft_model(finetune_data, finetune_target, test_data, test_target,
                   num_cols, cat_cols, cat_cardinalities, device,
                   d_token=96, num_layers=3, num_heads=4,
                   dim_feedforward=384, dropout=0.1, batch_size=256, epochs=200):
    """Train FT-Transformer on full finetune data and evaluate on test set"""

    # Create data loaders
    ft_loader, test_loader = create_ft_loaders(
        finetune_data, finetune_target, test_data, test_target,
        num_cols, cat_cols, batch_size
    )

    # Initialize model
    model = FTTransformer(
        n_num_features=len(num_cols),
        cat_cardinalities=cat_cardinalities,
        d_token=d_token,
        num_layers=num_layers,
        num_heads=num_heads,
        dim_feedforward=dim_feedforward,
        dropout=dropout
    ).to(device)

    # Optimizer, loss, and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    pos_weight = (finetune_target == 0).sum() / (finetune_target == 1).sum() if (finetune_target == 1).sum() > 0 else 1.0
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    # Training loop with early stopping
    best_loss = float('inf')
    patience_counter = 0
    loss_history = []

    print("\nTraining FT-Transformer...")
    for epoch in tqdm(range(1, epochs + 1), desc='Training'):
        avg_loss = train_epoch_ft(model, ft_loader, optimizer, criterion, device)
        loss_history.append(avg_loss)
        scheduler.step()

        # Early stopping
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1

        if patience_counter >= 20:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_model_state)

    # Evaluate on test set
    recall, f1, auprc, roc = evaluate_model_ft(model, test_loader, device)

    return {
        'test_recall': recall,
        'test_f1': f1,
        'test_auprc': auprc,
        'test_roc': roc,
        'model': model,
        'loss_history': loss_history
    }


# ============================================================
# RUN AND OUTPUT SUMMARY TABLE
# ============================================================

print("\n" + "="*60)
print("FT-TRANSFORMER")
print("="*60)

# Configuration parameters
d_token = 96
num_layers = 3
num_heads = 4
dim_feedforward = d_token * 4
dropout = 0.1
batch_size = 256
epochs = 200

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Cross-validation
cv_results = cross_validate_ft(
    finetune_data, finetune_target, num_cols, cat_cols, cat_cardinalities,
    device, d_token, num_layers, num_heads, dim_feedforward, dropout, batch_size, epochs=100
)

# Main training
test_results = train_ft_model(
    finetune_data, finetune_target, test_data, test_target,
    num_cols, cat_cols, cat_cardinalities, device,
    d_token, num_layers, num_heads, dim_feedforward, dropout, batch_size, epochs
)

# ============================================================
# SUMMARY TABLE
# ============================================================

print("\n" + "="*60)
print("SUMMARY METRICS TABLE")
print("="*60)

# Build FT-Transformer results row
ft_row = {
    'Model': 'FT-Transformer',
    'CV AUPRC': f"{cv_results['auprc']:.6f} ± {cv_results['auprc_std']:.6f}",
    'CV F1': f"{cv_results['f1']:.6f} ± {cv_results['f1_std']:.6f}",
    'CV Recall': f"{cv_results['recall']:.3f} ± {cv_results['recall_std']:.3f}",
    'Test AUPRC': f"{test_results['test_auprc']:.6f}",
    'Test F1': f"{test_results['test_f1']:.6f}",
    'Test Recall': f"{test_results['test_recall']:.2f}"
}

# Collect all results
results_list = [ft_row]

summary_ft = pd.DataFrame(results_list)
print(summary_ft.to_string(index=False))

# ============================================================
# LOSS VISUALIZATION
# ============================================================

plt.figure(figsize=(10, 6))
plt.plot(test_results['loss_history'], linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('FT-Transformer Training Loss', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()